# Module 3: Fashion Dataset Collection & Understanding
## Exploratory Data Analysis & Visual Catalog Inspection

This notebook demonstrates:
1. Loading the fashion catalog metadata and images using `FashionDataset`.
2. Inspecting category distributions, gender breakdown, and color palettes.
3. Visualizing garments across categories.
4. Generating and testing candidate outfit combinations.

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from PIL import Image

from src.dataset import FashionDataset

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

### 1. Initialize Dataset & Load Summary Statistics

In [ ]:
dataset = FashionDataset()
stats = dataset.get_summary_stats()
print(f"Total items loaded: {stats.get('total_items')}")
print(f"Images found on disk: {stats.get('images_found')}")
dataset.df.head()

### 2. Category & Outfit Part Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Category counts
cat_counts = dataset.df["canonical_category"].value_counts()
sns.barplot(x=cat_counts.values, y=cat_counts.index, ax=axes[0], palette="viridis")
axes[0].set_title("Garment Items by Category", fontsize=14, fontweight="bold")
axes[0].set_xlabel("Count")

# Outfit part distribution
part_counts = dataset.df["outfit_part"].value_counts()
sns.barplot(x=part_counts.values, y=part_counts.index, ax=axes[1], palette="mako")
axes[1].set_title("Outfit Part Distribution (Top/Bottom/Shoes/Accessories)", fontsize=14, fontweight="bold")
axes[1].set_xlabel("Count")

plt.tight_layout()
plt.show()

### 3. Visual Catalog Gallery: Sample Garments Across Categories

In [ ]:
categories = list(dataset.df["canonical_category"].unique())[:8]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for idx, cat in enumerate(categories):
    subset = dataset.get_items_by_category(cat, limit=1)
    if not subset.empty:
        item_id = subset.iloc[0]["id"]
        img = dataset.load_image(item_id, target_size=(224, 224))
        if img:
            axes[idx].imshow(img)
            axes[idx].set_title(f"{cat}\n(ID: {item_id})", fontsize=11, fontweight="bold")
    axes[idx].axis("off")

plt.suptitle("Sample Catalog Items Across Categories", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

### 4. Coordinated Candidate Outfit Generation

In [ ]:
outfit = dataset.sample_random_outfit(include_accessory=True)

parts = [p for p in ["top", "bottom", "shoes", "accessory"] if outfit.get(p) is not None]
fig, axes = plt.subplots(1, len(parts), figsize=(4 * len(parts), 5))
if len(parts) == 1:
    axes = [axes]

for idx, part in enumerate(parts):
    item = outfit[part]
    img = dataset.load_image(item["id"], target_size=(224, 224))
    if img:
        axes[idx].imshow(img)
        axes[idx].set_title(f"{part.upper()}\n{item.get('productDisplayName', 'Item')}", fontsize=11, fontweight="bold")
    axes[idx].axis("off")

plt.suptitle("Sample Coordinated Outfit Combination", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()